# 5.2 교차 검증과 그리드 서치

In [91]:
import pandas as pd
wine = pd.read_csv('https://bit.ly/wine_csv_data')

In [92]:
data = wine[['alcohol', 'sugar', 'pH']]
target = wine['class']

In [93]:
from sklearn.model_selection import train_test_split
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size=0.2, random_state=42)

In [94]:
sub_input, val_input, sub_target, val_target = train_test_split(train_input, train_target, test_size=0.2, random_state=42)

In [95]:
print(sub_input.shape, val_input.shape, test_input.shape)

(4157, 3) (1040, 3) (1300, 3)


In [96]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target)
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


In [97]:
from sklearn.model_selection import cross_validate
scores = cross_validate(dt, train_input, train_target)
print(scores)

{'fit_time': array([0.04009867, 0.03243637, 0.04842949, 0.03628945, 0.04313612]), 'score_time': array([0.00894094, 0.0076828 , 0.00614452, 0.00972867, 0.01618791]), 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}


In [98]:
import numpy as np
print(np.mean(scores['test_score']))

0.855300214703487


In [99]:
from sklearn.model_selection import StratifiedKFold
scores = cross_validate(dt, train_input, train_target, cv=StratifiedKFold())
print(np.mean(scores['test_score']))

0.855300214703487


In [100]:
from sklearn.model_selection import GridSearchCV
params = {'min_impurity_decrease': [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]}

In [101]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs = -1)

In [102]:
gs.fit(train_input, train_target)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'min_impurity_decrease': [0.0001, 0.0002, 0.0003,
                                                   0.0004, 0.0005]})

In [103]:
dt = gs.best_estimator_
print(dt.score(train_input, train_target))

0.9615162593804117


In [104]:
print(gs.best_params_)

{'min_impurity_decrease': 0.0001}


In [105]:
print(gs.cv_results_['mean_test_score'])

[0.86819297 0.86453617 0.86492226 0.86780891 0.86761605]


In [106]:
print(gs.cv_results_['params'][gs.best_index_])

{'min_impurity_decrease': 0.0001}


In [107]:
params = {'min_impurity_decrease': np.arange(0.0001, 0.001, 0.0001),
          'max_depth': range(5, 20, 1),
          'min_samples_split': range(2, 100, 10)}

In [108]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1)

In [109]:
from scipy.stats import uniform, randint

In [110]:
rgen = randint(0, 10)
rgen.rvs(10)

array([2, 8, 7, 6, 0, 3, 1, 9, 4, 3])

In [111]:
np.unique(rgen.rvs(1000), return_counts=True)

(array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([ 83, 106, 118, 108,  89,  95,  91, 107, 110,  93]))

In [112]:
ugen = uniform(0, 1)
ugen.rvs(10)

array([0.68836683, 0.52504759, 0.86282509, 0.48881574, 0.71966548,
       0.91597848, 0.35307363, 0.91652064, 0.94136978, 0.34573909])

In [113]:
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_sample_split': randint(2, 25),
          'min_sample_leaf': randint(1, 25)}

In [114]:
params

{'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen at 0x7b635cd84f20>,
 'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7b635cd8e8d0>,
 'min_sample_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7b635cd860f0>,
 'min_sample_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7b635cd861b0>}

In [115]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': randint(1, 25)}

rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, n_iter=100, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b635cd86c30>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7b635ca92a80>,
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b635cd86bd0>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b635cd86750>},
                   random_state=42)

In [116]:
print(rs.best_params_)

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [117]:
print(np.max(rs.cv_results_['mean_test_score']))

0.8695428296438884


In [118]:
dt = rs.best_estimator_
print(dt.score(test_input, test_target))

0.86


In [119]:
# 궁금증 : RandomizedSearchCV 에서 일부 변수만 랜덤 샘플링하는 것이 가능할까?
# 'min_samples_lear' 변수를 randint(1, 25) -> range(1, 25, 1) 로 바꿔보았다.

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': range(1, 25, 1)}


rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, n_iter=100, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b635cd368d0>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7b635d59e210>,
                                        'min_samples_leaf': range(1, 25),
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b635cd3c7d0>},
                   random_state=42)

In [120]:
# 생각보다 정상적으로 실행돼서 놀랐다. 아까와 같이 'min_samples_leaf'는 7이 나왔다. 그렇다면 간격을 1이 아니라 3으로 바꿔본다면?
print(rs.best_params_)

{'max_depth': 39, 'min_impurity_decrease': np.float64(0.00034102546602601173), 'min_samples_leaf': 7, 'min_samples_split': 13}


In [122]:
params = {'min_impurity_decrease': uniform(0.0001, 0.001),
          'max_depth': randint(20, 50),
          'min_samples_split': randint(2, 25),
          'min_samples_leaf': range(1, 25, 3)}


rs = RandomizedSearchCV(DecisionTreeClassifier(random_state=42), params, n_iter=100, n_jobs=-1, random_state=42)
rs.fit(train_input, train_target)

RandomizedSearchCV(estimator=DecisionTreeClassifier(random_state=42),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b635da65520>,
                                        'min_impurity_decrease': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7b636c68f920>,
                                        'min_samples_leaf': range(1, 25, 3),
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7b635e4988f0>},
                   random_state=42)

In [123]:
# 'min_samples_leaf'이 이번에는 4가 나온다. 어라, 근데 1부터 3씩의 간격이면 1, 4, 7, ... 에서 7이라는 최적의 변수를 고를 수 있었는데, 왜 4를 골랐을까?
# 자세히 보면 'min_samples_split'이 13에서 4로 바뀐걸 볼 수 있다. 한 변수의 최적값은 다른 변수에 의해서도 영향을 받으니까, 간격의 크기가 커지면서 다른 결과가 나온 것을 알 수 있다.
# 결론: RandomizedSearchCV 에 할당하는 변수는 randint, uniform과 같은 무작위 변수 샘플링이 아니더라도 range 같은 간격을 정할 수 있는 범위를 변수로 주어도 된다.
# 즉, 어떤 변수는 무작위로 추출해도 되고 어떤 변수는 내가 간격을 정해줘도 알아서 잘 돌아간다~
print(rs.best_params_)

{'max_depth': 31, 'min_impurity_decrease': np.float64(0.0003517822958253642), 'min_samples_leaf': 4, 'min_samples_split': 4}


In [124]:
dt = rs.best_estimator_
print(dt.score(test_input, test_target))

0.8607692307692307
